# Day 050 · scikit-learn 入门
**scikit-learn** · 阶段 P2 · Python 量化工具栈

> 很多人冲着一句话来学量化:听说用机器学习能预测股价、躺着就把钱赚了。这一节我们就请出最有名的机器学习工具 scikit-learn,亲手训练一个模型,用东方财富过去几天的样子去猜它明天涨还是跌。但我先把丑话说在前面:这节课的灵魂,是一个会让你清醒的反直觉真相。我们会老老实实地跑完整套流程,然后你会发现,模型的准确率往往只比抛硬币的50%高那么一丢丢。机器学习是个好工具,但它不是印钞机,别幻想靠它躺赢。带着这份诚实,我们一起把它跑明白。

---

**课件生成日期:** 2026-06-14  ·  **建议学习时长:** 22 分钟

学习路径建议:1)先看视频建立直觉 → 2)阅读本 notebook 跑代码 → 3)看 PDF 课件复习要点 → 4)做自测题

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有需要的 Python 包,缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续下面的代码

> 这一格只在第一次跑要等几十秒,后面再开 notebook 就秒过。

In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可) ===
# 检测缺失的库 → 自动 pip 安装 → 注入中文字体 → 一行命令搞定
import importlib
import subprocess
import sys
import os

REQUIRED = ["baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels", "yfinance"]
PIP_NAME = {
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
    "PIL": "Pillow",
    "bs4": "beautifulsoup4",
    "yaml": "PyYAML",
}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))

if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,正在自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置(让 matplotlib 不出乱码) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

CJK_FONT_PATHS = [
    "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",  # Linux/WSL
    "C:/Windows/Fonts/msyh.ttc",                               # Windows 微软雅黑
    "C:/Windows/Fonts/simhei.ttf",                             # Windows 黑体
    "/System/Library/Fonts/PingFang.ttc",                      # macOS 苹方
    "/System/Library/Fonts/STHeiti Medium.ttc",                # macOS 黑体
]
for p in CJK_FONT_PATHS:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP", "Microsoft YaHei",
                                    "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪 — 现在可以跑下面的代码单元格")


## 学习目标

- 理解机器学习最核心的套路:fit 喂历史数据让它学规律,predict 给新数据让它猜答案
- 搞清什么是特征(过去几天的线索)、什么是标签(明天涨还是跌的答案)
- 学会用 Pipeline 把标准化和模型打包,在交叉验证里自动防数据泄漏
- 明白股票是时间序列,只能拿过去训练、猜未来,绝不能像普通数据那样随机打乱
- 亲眼看到模型准确率约等于抛硬币,从此对各种AI选股稳赚的宣传有免疫力

## 历史背景:老韩信了AI选股稳赚的课,自己跑出来准确率才51%

老韩是个炒了多年股的散户,前阵子刷到一个课,标题写得震天响:用人工智能预测涨跌,准确率高得吓人,跟着买就稳赚。老韩心动了,交了学费,跟着学了一阵 scikit-learn。一开始他特别兴奋,觉得自己马上就要财富自由了。可等他自己照着把代码完整跑了一遍,用真实数据做了规规矩矩的时序交叉验证,屏幕上蹦出来的数字让他愣住了:平均准确率只有51%,比抛硬币的50%就高那么一点点。他反复检查代码,没错;换了几只股票,还是这样。老韩这才明白过来:那个课为了好卖,要么偷偷用了未来的信息让回测好看,要么压根没告诉你扣掉手续费之后这一丢丢优势根本不够赚钱。从那以后老韩想通了一件事:机器学习是个帮你看清世界的工具,不是一台印钞机。能不能赚钱,靠的是你对它的边界有没有敬畏,而不是幻想它替你躺赢。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. fit 和 predict:机器学习最核心的两步套路

整个 scikit-learn 用起来,核心就两个动作。fit 是训练,你把过去的历史数据喂给模型,跟它说这是输入、这是答案,它自己去找里面的规律。predict 是预测,规律学好之后,你给它一组新的输入,它就吐出一个猜测。打个比方:fit 就像给一个学生看一万道带答案的旧题让他总结规律,predict 就是考试时给他一道新题让他作答。你会发现几乎所有模型,不管多复杂,都是这一对 fit 加 predict,套路完全一样。


### 2. 特征和标签:线索是特征,答案是标签

要让模型学,得先告诉它拿什么当线索、猜什么当答案。线索就叫特征,答案就叫标签。在我们这节课里,特征是过去5天累计涨了多少、过去几天波动有多大、成交量比平时放大了多少这些看得见的线索;标签就是明天到底涨还是跌这一个答案,涨记成1、跌记成0。模型做的事,就是从一大堆历史的特征和标签里,试着找出线索和答案之间的关系。特征选得好不好,常常比用什么模型还重要。


### 3. Pipeline 加标准化:打包成流水线,顺便防泄漏

不同的特征量纲差很远,有的是百分之几、有的是几倍,直接喂给模型容易被大数字带偏。标准化就是把所有特征都拉到同一把尺子上,公平比较。Pipeline 的作用是把标准化这一步和模型这一步打包成一条流水线,一个 fit 全搞定。它还有个救命的好处:在交叉验证里,Pipeline 会自动保证标准化只用训练那部分数据来算尺子,绝不偷看测试数据,从结构上帮你堵住一个最隐蔽的坑。


### 4. TimeSeriesSplit:股票是时间序列,只能拿过去猜未来

普通的机器学习喜欢把数据随机打乱再切成几份,你训练我测试。但股票绝对不能这么干。股票是时间序列,有先后顺序,如果你随机打乱,就可能拿2024年的行情去训练、回头猜2020年的涨跌,等于让模型偷看了未来,跑出来的成绩漂亮得离谱却完全是假的。TimeSeriesSplit 专门解决这个问题:它永远拿前面一段时间训练、紧接着的后面一段测试,严格按时间顺序来,跑出来的成绩才是你实盘真能遇到的样子。


### 5. 数据泄漏:不小心用了未来信息,本课最大的坑

数据泄漏是机器学习里最隐蔽也最致命的坑,意思是你在训练时不小心用到了当时根本拿不到的未来信息。比如随机打乱时序、比如在全部数据上算标准化的尺子、比如造特征时混进了明天才知道的数字。一旦泄漏,回测曲线会美得让你心跳加速,可一上实盘立刻原形毕露、亏得你怀疑人生。这节课我们用 shift 和 rolling 严格保证每个特征只含当天及以前,再用 Pipeline 加 TimeSeriesSplit 双重防漏,就是为了让你养成对这个坑的肌肉记忆。


## 实操:scikit-learn 入门 — Pipeline+StandardScaler / LogisticRegression·RandomForest 预测涨跌 / TimeSeriesSplit 交叉验证 / 诚实揭示≈抛硬币

下面这段代码跟视频里讲解的 highlights 是一致的,可以**直接 Run All** 看结果。

**依赖安装:**
```bash
pip install pandas numpy matplotlib yfinance akshare statsmodels
```


In [ ]:
# day_050_sklearn_intro.py — scikit-learn 机器学习入门:用过去几天的样子猜明天涨还是跌
# 核心套路 fit 学规律 / predict 做预测,用 Pipeline 把标准化和模型打包成一条流水线
# 真名上屏:TimeSeriesSplit 时序交叉验证严防数据泄漏(data leakage),只能拿过去训练、猜未来
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import accuracy_score

# ==== 铁律62:数据缓存 helper —— 自适应定位仓库根 data/ 目录 ====
def _data_path(_name):
    # 铁律62:data/ 放在 notebook 文件夹里。仓库根(run_lab)存取 out/notebook/data/;
    # 原版 notebook 在 out/notebook/ 用 data/;中国版在 out/notebook/cn/ 用 ../data/
    from pathlib import Path as _P
    _here = _P.cwd()
    for _b in [_here/'data', _here/'..'/'data', _here/'out'/'notebook'/'data', _here/'..'/'..'/'data', _here/'..'/'..'/'..'/'data']:
        if (_b/_name).exists():
            return str(_b/_name)
    if (_here/'out'/'notebook').exists():
        _t = _here/'out'/'notebook'/'data'
    elif _here.name == 'cn':
        _t = _here/'..'/'data'
    else:
        _t = _here/'data'
    _t.mkdir(parents=True, exist_ok=True)
    return str(_t/_name)

pd.set_option('display.width', 160)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)
CODE, NAME = 'sz.300059', '东方财富'
START, END = '2019-01-01', '2024-12-31'

# ==== 0. 数据接入:CSV 缓存优先,没有才登录 baostock 联网拉取(铁律62)====
print('==== 0. 拉东方财富日线,准备数据 ====')
CSV = _data_path('D050_sklearn.csv')
import os
if os.path.exists(CSV):
    print(f'命中本地缓存 {CSV},直接读取')
    df = pd.read_csv(CSV, index_col=0, parse_dates=True)
else:
    print('本地无缓存,登录 baostock 联网拉取')
    lg = bs.login()
    if lg.error_code != '0':
        raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
    rs = bs.query_history_k_data_plus(CODE, 'date,open,high,low,close,volume',
                                      start_date=START, end_date=END, frequency='d')
    rows = []
    while rs.error_code == '0' and rs.next():
        rows.append(rs.get_row_data())
    bs.logout()
    df = pd.DataFrame(rows, columns=['date', 'open', 'high', 'low', 'close', 'volume'])
    df['date'] = pd.to_datetime(df['date'])
    for c in ['open', 'high', 'low', 'close', 'volume']:
        df[c] = pd.to_numeric(df[c])
    df = df.set_index('date').sort_index()
    df.to_csv(CSV)
    print(f'已缓存到 {CSV}')
print(f'{NAME} 共 {len(df)} 个交易日,{df.index.min().date()} 到 {df.index.max().date()}')

# ==== 1. 造特征:全部只用当天及以前的信息,严防偷看未来 ====
print('\n==== 1. 造 5 个特征(只含过去信息)====')
close = df['close']
daily_ret = close.pct_change()
feat = pd.DataFrame(index=df.index)
feat['ret_5d'] = close.pct_change(5)                              # 过去5日累计收益
feat['mom_10d'] = close.pct_change(10)                            # 过去10日动量
feat['vol_5d'] = daily_ret.rolling(5).std()                       # 过去5日波动率
feat['vol_ratio'] = df['volume'] / df['volume'].rolling(20).mean()  # 量比
feat['ma20_bias'] = close / close.rolling(20).mean() - 1          # 相对20日均线偏离
print('特征列:', list(feat.columns))

# ==== 2. 造标签:明天涨记 1、跌记 0 ====
print('\n==== 2. 造标签(明天涨=1 跌=0)====')
label = (close.pct_change().shift(-1) > 0).astype(int)
data = feat.copy()
data['label'] = label
data = data.dropna()
X = data[feat.columns].values
y = data['label'].values
print(f'样本数 {len(y)},其中明天涨的占比 {y.mean()*100:.1f}%(这就是抛硬币基准附近)')

# ==== 3. Pipeline:标准化 + 逻辑回归打包成一条流水线 ====
print('\n==== 3. Pipeline = StandardScaler + LogisticRegression ====')
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000)),
])
print('Pipeline 把标准化和模型打包,交叉验证时自动只用训练折拟合尺子,防数据泄漏')

# ==== 4. TimeSeriesSplit 时序交叉验证:只拿过去训练、未来测试 ====
print('\n==== 4. TimeSeriesSplit 交叉验证(绝不随机打乱)====')
tscv = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(pipe, X, y, cv=tscv, scoring='accuracy')
for i, s in enumerate(scores, 1):
    print(f'  第 {i} 折准确率:{s*100:.2f}%')
print(f'5 折平均准确率:{scores.mean()*100:.2f}%(对比抛硬币基准 50%)')
print(f'比抛硬币高 {(scores.mean()-0.5)*100:+.2f} 个百分点 , 往往只高一丢丢')

# ==== 5. 随机森林看特征重要性:哪个线索最有用 ====
print('\n==== 5. RandomForest 特征重要性 ====')
n_train = int(len(X) * 0.8)
X_tr, X_te = X[:n_train], X[n_train:]
y_tr, y_te = y[:n_train], y[n_train:]
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)
imp = pd.Series(rf.feature_importances_, index=feat.columns).sort_values(ascending=False)
print(imp.to_string())
print(f'最有用的特征是:{imp.index[0]}(重要性 {imp.iloc[0]:.3f})')
rf_acc = accuracy_score(y_te, rf.predict(X_te))
print(f'随机森林在测试段准确率:{rf_acc*100:.2f}%')

# ==== 6. 诚实的策略对照:按模型信号操作 vs 一直持有 ====
print('\n==== 6. 诚实对照:模型择时 vs 一直持有 ====')
pipe.fit(X_tr, y_tr)
pred = pipe.predict(X_te)
test_idx = data.index[n_train:]
next_ret = close.pct_change().shift(-1).reindex(test_idx).fillna(0).values
strat_ret = np.where(pred == 1, next_ret, 0.0)   # 预测涨就持有,预测跌就空仓
nav_strat = np.cumprod(1 + strat_ret)
nav_hold = np.cumprod(1 + next_ret)
print(f'按模型信号操作 累计收益:{(nav_strat[-1]-1)*100:+.1f}%')
print(f'一直持有       累计收益:{(nav_hold[-1]-1)*100:+.1f}%')
print('两者大概率半斤八两甚至更差,还没扣手续费 , 机器学习不是印钞机')

# ==== 7. 画三张图 ====
print('\n==== 7. 画三张图:交叉验证 / 特征重要性 / 净值对照 ====')
# 图1:5 折交叉验证准确率 + 抛硬币基准线
plt.figure(figsize=(11, 6))
xpos = np.arange(1, len(scores) + 1)
plt.bar(xpos, scores * 100, color='#4c78a8', alpha=0.85, label='每折准确率')
plt.axhline(50, color='#d62728', ls='--', lw=2, label='抛硬币基准 50%')
plt.title(f'{NAME} 涨跌预测:5 折时序交叉验证准确率 vs 抛硬币')
plt.xlabel('第几折'); plt.ylabel('准确率(%)'); plt.xticks(xpos)
plt.ylim(40, 60); plt.legend(); plt.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig('cv_accuracy.png', dpi=120); plt.close()

# 图2:特征重要性
plt.figure(figsize=(11, 6))
imp_sorted = imp.sort_values()
plt.barh(imp_sorted.index, imp_sorted.values, color='#59a14f', alpha=0.85)
plt.title(f'{NAME} 随机森林特征重要性:哪个线索最有用')
plt.xlabel('重要性'); plt.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.savefig('feature_importance.png', dpi=120); plt.close()

# 图3:测试段净值对照
plt.figure(figsize=(11, 6))
plt.plot(test_idx, nav_strat, color='#4c78a8', lw=2, label='按模型信号操作')
plt.plot(test_idx, nav_hold, color='#e45756', lw=2, label='一直持有')
plt.title(f'{NAME} 测试段净值:模型择时 vs 一直持有(诚实对照)')
plt.xlabel('日期'); plt.ylabel('累计净值'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('strategy_nav.png', dpi=120); plt.close()

print('\n[done] fit/predict + Pipeline + TimeSeriesSplit + 特征重要性 + 诚实对照完成,3 张图已生成')

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 个股 | sz.300059 | 东方财富6年日线,造5个只含过去信息的特征预测明天涨跌,时序交叉验证看准确率 |
| 风险认知 |  | 用 TimeSeriesSplit 跑出平均准确率约51%,亲眼看到只比抛硬币的50%高一丢丢 |
| 特征分析 |  | 用随机森林的 feature_importances_ 看5个特征里哪个线索对预测涨跌最有用 |
| 策略对照 |  | 按模型信号择时 vs 一直持有,对比累计净值,大概率半斤八两还没扣手续费 |


## 常见坑

### ⚠ 01. 数据泄漏:用了未来信息或在全量上算标准化

这是机器学习最致命的坑。如果你在全部数据上算标准化的尺子、或者造特征时混进了明天才知道的数字,就等于让模型偷看了未来。回测曲线会美得让你心跳加速,一上实盘立刻原形毕露。务必用 Pipeline 让标准化只在训练折上拟合,造特征只用 shift 和 rolling 取当天及以前。

### ⚠ 02. 把时间序列随机打乱来切训练测试

普通机器学习喜欢随机打乱数据,但股票绝对不能这么干。随机打乱可能拿2024年的行情训练、回头猜2020年的涨跌,等于让模型偷看未来,成绩假到离谱。股票必须用 TimeSeriesSplit,永远拿过去训练、未来测试,跑出来的成绩才是你实盘真能遇到的。

### ⚠ 03. 过拟合:训练集上好得不行,测试集上原形毕露

模型如果太复杂、参数太多,它会把训练数据里的噪音都背下来,在训练集上准确率高得吓人,可一换到没见过的测试数据立刻崩盘。一定要看测试集和交叉验证的成绩,而不是训练集的成绩。训练好测试崩,就是典型的过拟合信号。

### ⚠ 04. 把51%准确率当成能赚钱

就算模型准确率有51%,比抛硬币高1个百分点,也别急着高兴。这一丢丢优势扣掉买卖的手续费和印花税之后,很可能根本不够覆盖成本,甚至倒亏。准确率高一点点不等于能赚钱,中间隔着交易成本这道坎,老韩就是栽在这里。

### ⚠ 05. 特征里混进了未来信息

造特征时最容易不知不觉用到未来。比如算波动率时不小心把明天的数据也算进去了,或者标签和特征对齐时错位了一天。每造一个特征都要反问自己:这个数字在当天收盘那一刻,我真的能拿到吗?拿不到的,就是泄漏。

## 实战 SOP · 用机器学习预测股票的几条规矩

1. 牢记 fit 学规律、predict 做预测的核心套路,几乎所有模型都是这一对
2. 股票是时间序列,只能用 TimeSeriesSplit 拿过去训练、未来测试,绝不随机打乱
3. 用 Pipeline 把标准化和模型打包,让标准化只在训练折拟合,从结构上防数据泄漏
4. 造特征只用当天及以前的信息,用 shift 和 rolling 严防偷看未来
5. 对准确率保持诚实:约等于抛硬币是常态,高一点点也未必能覆盖手续费

> 把这段打印贴在你电脑边,执行 1000 次它会回报你。

## 总结 · 你应该带走的

2. 机器学习的核心就两步:fit 喂历史让模型学规律,predict 给新数据让它猜答案。
3. 特征是过去几天的线索,标签是明天涨还是跌的答案,模型学的是两者的关系。
4. Pipeline 把标准化和模型打包成流水线,在交叉验证里自动防数据泄漏。
5. 股票是时间序列,必须用 TimeSeriesSplit 拿过去训练、未来测试,不能随机打乱。
6. 诚实的真相:模型准确率往往约等于抛硬币,只比50%高一丢丢。
7. 机器学习是工具不是印钞机,数据泄漏会让回测好看实盘崩,要对它的边界有敬畏。

## 自测题

**Q1.** 用大白话解释 fit 和 predict 各是干什么的,它们和考前刷题、考场答题有什么像的地方。

**Q2.** 什么是特征、什么是标签?在我们这节课里,特征和标签分别是哪些东西?

**Q3.** 为什么股票不能像普通数据那样随机打乱来切训练测试?TimeSeriesSplit 是怎么做的?

**Q4.** 老韩自己跑出来准确率只有51%,这说明了什么?为什么51%也不一定能赚钱?

把答案写下来,3 天后再回看。

## 下一节预告

**Day 051 · 装饰器与上下文管理器** (Decorators & ContextManagers)

亲手跑通机器学习、也认清了它的边界之后,下一节我们回到 Python 本身,学两个让代码瞬间变优雅的魔法:装饰器和上下文管理器。用它们给函数加外挂,自动计时、失败自动重试、结果自动缓存,还能优雅地管理文件和数据库连接,写出更专业、更省心的量化代码。

## 推荐阅读

- Géron《Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow》(2022/O'Reilly)— 用 scikit-learn 入门机器学习最受欢迎的实战教材,fit/predict/Pipeline 讲得最清楚。
- Pedregosa et al.《Scikit-learn: Machine Learning in Python》(2011/JMLR)— scikit-learn 官方论文,这套工具的来历和设计理念都在这里。
- Lopez de Prado《Advances in Financial Machine Learning》(2018/Wiley)— 专讲金融机器学习陷阱,数据泄漏、时序交叉验证、为什么大多数策略是假的。
- Hastie, Tibshirani & Friedman《The Elements of Statistical Learning》(2009/Springer)— 统计学习的经典权威教材,逻辑回归、随机森林、过拟合的理论根基。
- scikit-learn 官方文档(scikit-learn.org)— Pipeline、StandardScaler、TimeSeriesSplit、cross_val_score 的函数手册和示例。